# ProSe-Custody-Strategy

**Custody case strategy advisor — best interest factors, parenting plans, and trial preparation**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Roderick3rd/sovereign-training-notebooks/blob/main/ProSe-Custody-Strategy.ipynb)

## Description
Fine-tune a Gemma 4 9B model on legal domain data using Unsloth QLoRA. Trains on a free T4 GPU in Google Colab.

## Output
- Pushes trained LoRA adapter to HuggingFace: `Roderick3rd/ProSe-Custody-Strategy`
- Ready to use with transformers or Ollama


## 1. Install Dependencies
Run this cell to install Unsloth and HuggingFace Hub.


In [ ]:
!pip install unsloth -q
!pip install huggingface_hub -q
!pip install datasets -q
!pip install trl -q
!pip install accelerate -q
!pip install bitsandbytes -q


## 2. Login to HuggingFace
Paste your HuggingFace write token when prompted. Get one at https://huggingface.co/settings/tokens


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 3. Load Base Model (4-bit)
Loading `unsloth/gemma-4-9b-it-bnb-4bit` with Unsloth's 4-bit optimization for T4 GPU.


In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Model loaded: {{model.config._name_or_path}}")
print(f"Parameters: {{sum(p.numel() for p in model.parameters()):,}}")


## 4. Configure LoRA
Apply LoRA adapters to attention layers.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
print(f"Trainable params: {{sum(p.numel() for p in model.parameters() if p.requires_grad):,}}")


## 5. Load Dataset
Loading `Roderick3rd/hcjf-legal-forms` from HuggingFace.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="https://huggingface.co/datasets/Roderick3rd/hcjf-legal-forms/raw/main/ollama_training_qa.jsonl", split="train")
print(f"Dataset loaded: {{len(dataset)}} examples")
print(f"Columns: {{dataset.column_names}}")
print(f"First example keys: {{list(dataset[0].keys())}}")


## 6. Format Dataset
Convert to instruction format for training.


In [ ]:
def format_chat(example):
    """Convert JSONL to chat format."""
    if 'messages' in example:
        # Already in ChatML format
        text = tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False
        )
    elif 'instruction' in example and 'output' in example:
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": example['instruction']},
            {"role": "assistant", "content": example['output']},
        ], tokenize=False, add_generation_prompt=False)
    elif 'question' in example and 'answer' in example:
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']},
        ], tokenize=False, add_generation_prompt=False)
    else:
        text = str(example)
    return {"text": text}

formatted = dataset.map(format_chat)
print(f"Formatted: {{len(formatted)}} examples")
print(f"Sample:\n{{formatted[0]['text'][:300]}}")


## 7. Train
Run SFT training with Unsloth's optimized trainer.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete! Loss: {{trainer_stats.training_loss:.4f}}")


## 8. Save & Push to HuggingFace
Push the trained LoRA adapter to `Roderick3rd/ProSe-Custody-Strategy`.


In [ ]:
model_name = "Roderick3rd/ProSe-Custody-Strategy"

model.push_to_hub(model_name, tokenizer=tokenizer)
tokenizer.push_to_hub(model_name)
print(f"Model pushed to: {{model_name}}")


## 9. Test Inference
Quick test to verify the model works.


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a custody case strategy advisor. You analyze parenting situations using Ohio's best interest factors (ORC 3109.04). You help parents prepare parenting plans, understand GAL recommendations, and prepare for custody hearings. You are thorough, evidence-focused, and cite Ohio law."},
    {{"role": "user", "content": "What forms do I need to file for custody in Hamilton County Juvenile Court?"}},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)
